# Session 7.5: Medical Image Classification - Pneumonia Detection

**Objectives:** Build healthcare CV model, handle class imbalance, achieve >85% recall
**Dataset:** Chest X-Ray Images (Pneumonia)
**Time:** 90-120 minutes

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.applications import VGG16
from tensorflow.keras import layers, models
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from sklearn.metrics import classification_report, roc_curve, auc, confusion_matrix
import seaborn as sns

np.random.seed(42)
tf.random.set_seed(42)

## 1. Load Chest X-Ray Dataset

**Directory structure:**
```
chest_xray/
  train/
    NORMAL/
    PNEUMONIA/
  test/
    NORMAL/
    PNEUMONIA/
```

In [ ]:
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    width_shift_range=0.1,
    height_shift_range=0.1,
    zoom_range=0.1,
    horizontal_flip=True
)

test_datagen = ImageDataGenerator(rescale=1./255)

train_gen = train_datagen.flow_from_directory(
    'chest_xray/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

test_gen = test_datagen.flow_from_directory(
    'chest_xray/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

print(f'Training samples: {train_gen.samples}')
print(f'Test samples: {test_gen.samples}')
print(f'Classes: {train_gen.class_indices}')

## 2. Handle Class Imbalance

In [ ]:
from sklearn.utils import class_weight

class_weights = class_weight.compute_class_weight(
    'balanced',
    classes=np.unique(train_gen.classes),
    y=train_gen.classes
)
class_weights_dict = dict(enumerate(class_weights))

print(f'Class weights: {class_weights_dict}')

## 3. Build Transfer Learning Model

In [ ]:
base_model = VGG16(weights='imagenet', include_top=False, input_shape=(224, 224, 3))
base_model.trainable = False

model = models.Sequential([
    base_model,
    layers.GlobalAveragePooling2D(),
    layers.Dense(256, activation='relu'),
    layers.Dropout(0.5),
    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),
    layers.Dense(1, activation='sigmoid')
])

model.compile(
    optimizer='adam',
    loss='binary_crossentropy',
    metrics=['accuracy', tf.keras.metrics.Recall(), tf.keras.metrics.Precision()]
)

model.summary()

## 4. Train Model

In [ ]:
history = model.fit(
    train_gen,
    epochs=15,
    validation_data=test_gen,
    class_weight=class_weights_dict,
    callbacks=[
        tf.keras.callbacks.EarlyStopping(patience=5, restore_best_weights=True),
        tf.keras.callbacks.ModelCheckpoint('pneumonia_model.h5', save_best_only=True)
    ]
)

print('Training completed')

## 5. Medical Evaluation (Recall is Critical!)

In [ ]:
y_pred_proba = model.predict(test_gen)
y_pred = (y_pred_proba > 0.5).astype(int)
y_true = test_gen.classes

print('Classification Report:')
print(classification_report(y_true, y_pred, target_names=['Normal', 'Pneumonia']))

recall = tf.keras.metrics.Recall()
recall.update_state(y_true, y_pred)
recall_score = recall.result().numpy()

print(f'\nRecall: {recall_score:.4f}')
if recall_score > 0.85:
    print('✓ SUCCESS: Recall >85% - Good at catching pneumonia cases!')
else:
    print('⚠ Recall below target. Consider adjusting threshold or retraining.')

## 6. ROC Curve

In [ ]:
fpr, tpr, thresholds = roc_curve(y_true, y_pred_proba)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='darkorange', lw=2, label=f'ROC (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='navy', lw=2, linestyle='--')
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate (Recall)')
plt.title('ROC Curve - Pneumonia Detection', fontweight='bold')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.show()

## 7. Confusion Matrix

In [ ]:
cm = confusion_matrix(y_true, y_pred)

plt.figure(figsize=(8, 6))
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', 
            xticklabels=['Normal', 'Pneumonia'],
            yticklabels=['Normal', 'Pneumonia'])
plt.title('Confusion Matrix', fontweight='bold')
plt.ylabel('True Label')
plt.xlabel('Predicted Label')
plt.show()

tn, fp, fn, tp = cm.ravel()
print(f'\nTrue Negatives: {tn}')
print(f'False Positives: {fp}')
print(f'False Negatives: {fn} ← Critical to minimize!')
print(f'True Positives: {tp}')

## Key Takeaways

### Medical Imaging Considerations:
- **Recall > Precision:** Missing pneumonia is worse than false alarm
- **Class Imbalance:** Use class weights
- **ROC/AUC:** Better metric than accuracy
- **Interpretability:** Grad-CAM helps validate predictions
- **Regulatory:** FDA approval requires extensive validation

### Clinical Impact:
- False Negative (FN): Missed pneumonia case - DANGEROUS
- False Positive (FP): Unnecessary treatment - Less critical
- Target: Maximize Recall (sensitivity)

**Session Complete!**